In [51]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [52]:
from IPython.core.display import display, HTML, clear_output
display(HTML('<style>.container { width:90% !important; }</style>'))
display(HTML('<style>.prompt { min-width:10ex !important; }</style>'))
display(HTML('<style>div#notebook { font-size:12px !important; }</style>'))

/tmp/ipykernel_3717/4223674425.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML, clear_output
/tmp/ipykernel_3717/4223674425.py:1: DeprecationWarning: Importing clear_output from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML, clear_output


In [53]:
import pyspark

In [54]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("test_app_mazafaka") \
    .getOrCreate()

In [55]:
spark

In [56]:
import os
csv_path = '/home/jovyan/.cache/kagglehub/datasets/akalyasubramanian/dataset-for-spark-df-operations/versions/1/athletes.csv'
if not os.path.exists(csv_path):
    ! pip install -q kagglehub
    import kagglehub
    kagglehub.dataset_download('akalyasubramanian/dataset-for-spark-df-operations')
! hdfs dfs -mkdir -p /user/spark_advanced
! hdfs dfs -test -e /user/spark_advanced/athletes.csv || hadoop fs -put {csv_path} /user/spark_advanced/

In [57]:
! hdfs dfs -text /user/spark_advanced/athletes.csv | head -n 20





















text: Unable to write to output stream.


In [58]:
athletes = spark.read.csv('/user/spark_advanced/athletes.csv', header=True, inferSchema=True)

# RDD — Resilient Distributed Dataset
RDD — это базовая абстракция Spark: распределённая коллекция элементов, обрабатываемая параллельно на нескольких узлах кластера.

**Ключевые свойства RDD:**
- **Resilient** — устойчивость: при сбое узла данные восстанавливаются из lineage (графа преобразований)
- **Distributed** — данные распределены по партициям на разных узлах
- **Immutable** — нельзя изменить: каждое преобразование создаёт новый RDD
- **Lazy** — преобразования (transformations) выполняются только при вызове действия (action)

**Когда использовать RDD вместо DataFrame?**
- Неструктурированные данные (текст, логи сложной структуры)
- Нужен тонкий контроль над физическим выполнением (partitioning)
- Нужно работать с legacy-кодом

В большинстве случаев **DataFrame эффективнее** (Catalyst optimizer + Tungsten).

## Способы создания RDD
1. `spark.sparkContext.parallelize(collection)` — из локальной коллекции Python
2. `spark.sparkContext.textFile(path)` — из текстового файла (HDFS, S3, локальный)
3. `df.rdd` — конвертация из DataFrame
4. `rdd1.union(rdd2)` — объединение существующих RDD

In [59]:
# Способ 1: parallelize — из Python-коллекции
sc = spark.sparkContext

rdd_from_list = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print("RDD из списка:", rdd_from_list.collect())

# Можно задать количество партиций
rdd_partitioned = sc.parallelize(range(100), numSlices=5)
print("Количество партиций:", rdd_partitioned.getNumPartitions())

# Строки
rdd_words = sc.parallelize(["spark", "hadoop", "kafka", "spark", "flink"])
print("RDD строк:", rdd_words.collect())

RDD из списка: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Количество партиций: 5
RDD строк: ['spark', 'hadoop', 'kafka', 'spark', 'flink']


In [60]:
# Способ 2: textFile — чтение текстового файла
# Создадим временный файл и прочитаем его
import tempfile, os
tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
tmp.writelines(["Apache Spark is a unified analytics engine\n",
                "for large-scale data processing\n",
                "Spark provides high-level APIs\n"])
tmp.close()

rdd_text = sc.textFile("file://" + tmp.name)
for line in rdd_text.collect():
    print(repr(line))
os.unlink(tmp.name)

'Apache Spark is a unified analytics engine'
'for large-scale data processing'
'Spark provides high-level APIs'


In [61]:
# Способ 3: из DataFrame
athletes = spark.read.csv("/user/spark_advanced/athletes.csv", header=True, inferSchema=True)
athletes_rdd = athletes.rdd

print("Тип:", type(athletes_rdd))
print("Количество партиций:", athletes_rdd.getNumPartitions())
print("Первые 3 элемента:")
for row in athletes_rdd.take(3):
    print("  ", row)

Тип: <class 'pyspark.rdd.RDD'>
Количество партиций: 9
Первые 3 элемента:
   Row(ID=1, Name='A Dijiang', Gender='M', Age='24', Height='180', Weight='80', Team='China', NOC='CHN', Games='1992 Summer', Year=1992, Season='Summer', City='Barcelona', Sport='Basketball', Event="Basketball Men's Basketball", Medal='NA')
   Row(ID=2, Name='A Lamusi', Gender='M', Age='23', Height='170', Weight='60', Team='China', NOC='CHN', Games='2012 Summer', Year=2012, Season='Summer', City='London', Sport='Judo', Event="Judo Men's Extra-Lightweight", Medal='NA')
   Row(ID=3, Name='Gunnar Nielsen Aaby', Gender='M', Age='24', Height='NA', Weight='NA', Team='Denmark', NOC='DEN', Games='1920 Summer', Year=1920, Season='Summer', City='Antwerpen', Sport='Football', Event="Football Men's Football", Medal='NA')


## Transformations vs Actions

**Transformations (преобразования)** — создают новый RDD, выполняются **лениво** (не запускаются сразу):
`map`, `flatMap`, `filter`, `distinct`, `sample`, `union`, `intersection`, `sortBy`, `reduceByKey`, `groupByKey`, `join`, `cartesian`, `repartition`, `coalesce` ...

**Actions (действия)** — запускают выполнение графа и возвращают результат драйверу или сохраняют данные:
`collect`, `take`, `count`, `first`, `reduce`, `fold`, `aggregate`, `foreach`, `saveAsTextFile`, `top`, `takeOrdered` ...

До вызова action никаких вычислений не происходит — Spark только строит lineage-граф.

In [67]:
rdd = sc.parallelize([1, 2, 3, 4, 6])

In [62]:
# Демонстрация ленивости: transformation не выполняется сразу
rdd = sc.parallelize([1, 2, 3, 4, 5])

print("1. Создали RDD")
mapped = rdd.map(lambda x: x * 10)       # transformation — ничего не вычисляется
print("2. Применили map — ничего не произошло")
filtered = mapped.filter(lambda x: x > 20)  # transformation — тоже ничего
print("3. Применили filter — тоже ничего")

result = filtered.collect()  # ACTION — вот тут запускается весь граф
print("4. collect() запустил вычисления. Результат:", result)

1. Создали RDD
2. Применили map — ничего не произошло
3. Применили filter — тоже ничего
4. collect() запустил вычисления. Результат: [30, 40, 50]


In [68]:
result = filtered.collect() 
print(result)

[30, 40, 50]


## Базовые transformations
### `map` — применить функцию к каждому элементу
Один элемент на входе → один элемент на выходе.

In [32]:
rdd = sc.parallelize([1, 2, 3, 4, 5])

# map: каждый элемент → x²
squared = rdd.map(lambda x: x ** 2)
print("Квадраты:", squared.collect())

# map к строкам
rdd_str = sc.parallelize(["hello", "world", "spark"])
upper_rdd = rdd_str.map(lambda s: s.upper())
print("Верхний регистр:", upper_rdd.collect())

# map: извлечение поля из Row
ages = athletes_rdd.map(lambda row: row.Age)
print("Первые 10 возрастов:", ages.take(10))

Квадраты: [1, 4, 9, 16, 25]
Верхний регистр: ['HELLO', 'WORLD', 'SPARK']
Первые 10 возрастов: ['24', '23', '24', '34', '21', '21', '25', '25', '27', '27']


### `flatMap` — map + "расплющить" результат
Один элемент на входе → **0 или больше** элементов на выходе.
Полезно для разбиения строк на слова, токенизации и т.д.

In [70]:
# flatMap: разбить строки на слова
sentences = sc.parallelize([
    "Apache Spark is fast",
    "Spark is a unified engine",
    "RDD is the low-level API"
])

# map вернёт списки
print("map (списки):", sentences.map(lambda s: s.split(" ")).collect())

# flatMap "расплющит" списки в один RDD
print("flatMap (слова):", sentences.flatMap(lambda s: s.split(" ")).collect())

# Подсчёт слов через flatMap + map + reduceByKey
word_counts = sentences \
    .flatMap(lambda s: s.split(" ")) \
    .map(lambda w: (w, 1)) \
    .reduceByKey(lambda a, b: a + b)
print("Частота слов:", word_counts.collect())

map (списки): [['Apache', 'Spark', 'is', 'fast'], ['Spark', 'is', 'a', 'unified', 'engine'], ['RDD', 'is', 'the', 'low-level', 'API']]
flatMap (слова): ['Apache', 'Spark', 'is', 'fast', 'Spark', 'is', 'a', 'unified', 'engine', 'RDD', 'is', 'the', 'low-level', 'API']
Частота слов: [('Apache', 1), ('unified', 1), ('engine', 1), ('a', 1), ('RDD', 1), ('API', 1), ('Spark', 2), ('low-level', 1), ('fast', 1), ('the', 1), ('is', 3)]


### `filter` — оставить только элементы, удовлетворяющие условию

In [71]:
# Фильтрация чисел
rdd = sc.parallelize(range(1, 21))

evens = rdd.filter(lambda x: x % 2 == 0)
print("Чётные:", evens.collect())

gt10 = rdd.filter(lambda x: x > 10)
print("Больше 10:", gt10.collect())

# Фильтрация athletes: только медалисты
medalists = athletes_rdd.filter(lambda row: row.Medal is not None and row.Medal != "NA")
print("Всего атлетов:", athletes_rdd.count())
print("Медалистов:", medalists.count())

NameError: name 'x' is not defined

### `distinct` — убрать дубликаты
Внимание: `distinct` требует shuffle по всему кластеру — дорого для больших данных.

In [35]:
rdd = sc.parallelize([1, 1, 2, 2, 2, 3, 4, 4, 5])
print("Уникальные:", rdd.distinct().collect())

# Уникальные виды спорта
sports = athletes_rdd.map(lambda row: row.Sport).distinct()
print("Уникальные виды спорта:", sports.count())
print(sports.take(10))

Уникальные: [1, 2, 3, 4, 5]
Уникальные виды спорта: 66
['Weightlifting', 'Tennis', 'Archery', 'Racquets', 'Art Competitions', 'Luge', 'Equestrianism', 'Trampolining', 'Triathlon', 'Curling']


### `sample` — случайная выборка
`sample(withReplacement, fraction, seed)`:
- `withReplacement` — может ли элемент попасть в выборку несколько раз
- `fraction` — ожидаемая доля элементов (не точный размер!)

In [72]:
rdd = sc.parallelize(range(1, 101))

# Без возвращения: ~20% элементов
sample1 = rdd.sample(False, 0.2, seed=42)
print("Sample 20% (без возвращения):", len(sample1.collect()), "элементов")

# С возвращением: ~20% элементов (могут повторяться)
sample2 = rdd.sample(True, 0.2, seed=42)
print("Sample 20% (с возвращением):", len(sample2.collect()), "элементов")

# takeSample — точное количество элементов (action, не transformation)
exact_sample = rdd.takeSample(False, 5, seed=42)
print("Точные 5 элементов:", exact_sample)

Sample 20% (без возвращения): 28 элементов
Sample 20% (с возвращением): 17 элементов
Точные 5 элементов: [75, 15, 30, 48, 74]


## Pair RDD (парный RDD)
Pair RDD — это RDD, где каждый элемент — кортеж `(key, value)`. Это аналог Map в Java или dict в Python.

Pair RDD поддерживают специальные операции: `reduceByKey`, `groupByKey`, `mapValues`, `keys`, `values`, `join`, `cogroup` и др.

In [73]:
# Создание Pair RDD: (вид_спорта, 1) для подсчёта
pair_rdd = athletes_rdd.map(lambda row: (row.Sport, 1))

# keys / values — извлечение ключей и значений
print("Первые 5 ключей:", pair_rdd.keys().take(5))
print("Первые 5 значений:", pair_rdd.values().take(5))

# mapValues — применить функцию ТОЛЬКО к значению (ключ не меняется)
scaled = pair_rdd.mapValues(lambda v: v * 100)
print("mapValues:", scaled.take(5))

# flatMapValues — аналог flatMap, но для значений Pair RDD
pairs = sc.parallelize([("a", [1, 2, 3]), ("b", [4, 5])])
print("flatMapValues:", pairs.flatMapValues(lambda v: v).collect())

Первые 5 ключей: ['Basketball', 'Judo', 'Football', 'Tug-Of-War', 'Speed Skating']
Первые 5 значений: [1, 1, 1, 1, 1]
mapValues: [('Basketball', 100), ('Judo', 100), ('Football', 100), ('Tug-Of-War', 100), ('Speed Skating', 100)]
flatMapValues: [('a', 1), ('a', 2), ('a', 3), ('b', 4), ('b', 5)]


### `reduceByKey` — агрегация по ключу
Самый эффективный способ агрегации в RDD: shuffle происходит **после** локальной агрегации на каждой партиции (combine).

**`reduceByKey` vs `groupByKey`:**
- `reduceByKey` — сначала локально объединяет значения на каждой партиции, потом shuffle. **Эффективно.**
- `groupByKey` — shuffle ВСЕХ значений по сети, потом группировка. **Неэффективно для больших данных.**

In [38]:
# reduceByKey: подсчёт атлетов по полу
gender_counts = athletes_rdd \
    .map(lambda row: (row.Gender, 1)) \
    .reduceByKey(lambda a, b: a + b)
print("Атлеты по полу:", gender_counts.collect())

# Топ-5 самых популярных видов спорта
top_sports = athletes_rdd \
    .map(lambda row: (row.Sport, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .takeOrdered(5, key=lambda x: -x[1])
print("Топ-5 видов спорта:", top_sports)

# Распределение атлетов по командам (NOC) с количеством более 1000
big_teams = athletes_rdd \
    .map(lambda row: (row.NOC, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] > 1000) \
    .takeOrdered(10, key=lambda x: -x[1])
print("Команды >1000 атлетов:", big_teams)

Атлеты по полу: [('M', 196594), ('F', 74522)]
Топ-5 видов спорта: [('Athletics', 38624), ('Gymnastics', 26707), ('Swimming', 23195), ('Shooting', 11448), ('Cycling', 10859)]
Команды >1000 атлетов: [('USA', 18853), ('FRA', 12758), ('GBR', 12256), ('ITA', 10715), ('GER', 9830), ('CAN', 9733), ('JPN', 8444), ('SWE', 8339), ('AUS', 7638), ('HUN', 6607)]


### `groupByKey` — группировка значений по ключу
Возвращает RDD вида `(key, [val1, val2, ...])`. Внимание: `groupByKey` может вызвать OOM, если одному ключу соответствует слишком много значений, т.к. все значения собираются в один список на одном узле.
В большинстве случаев `reduceByKey` или `aggregateByKey` — лучший выбор.

In [39]:
# groupByKey: сгруппировать медали по странам (NOC)
# Берём только медалистов и небольшую выборку для демонстрации
small_rdd = athletes_rdd \
    .filter(lambda row: row.Medal is not None and row.Medal != "NA") \
    .map(lambda row: (row.NOC, row.Medal)) \
    .sample(False, 0.01, seed=42)  # 1% данных для скорости

grouped = small_rdd.groupByKey()
# values — это итератор, нужно преобразовать в list для просмотра
result = grouped.mapValues(lambda vals: list(vals)).take(5)
for noc, medals in result:
    print(f"  {noc}: {medals[:10]}...")

# Тот же результат через reduceByKey — эффективнее!
# Но если нужно сохранить список значений (а не агрегат), groupByKey уместен

  SWE: ['Bronze', 'Bronze', 'Silver', 'Bronze', 'Bronze', 'Gold', 'Silver', 'Gold', 'Silver', 'Gold']...
  RUS: ['Gold', 'Gold', 'Gold', 'Silver', 'Silver', 'Gold', 'Bronze', 'Gold', 'Silver', 'Bronze']...
  SVK: ['Gold']...
  NGR: ['Bronze']...
  KEN: ['Gold']...


### `aggregateByKey` — гибкая агрегация по ключу
`aggregateByKey(zeroValue, seqFunc, combFunc)`:
- `zeroValue` — начальное значение для каждой партиции
- `seqFunc` — функция, объединяющая значение с аккумулятором на партиции
- `combFunc` — функция, объединяющая аккумуляторы с разных партиций

Позволяет вычислять несколько агрегаций одновременно.

In [40]:
# aggregateByKey: для каждой страны — (количество медалей, список уникальных видов спорта)
# Берём подвыборку для скорости
medal_rdd = athletes_rdd \
    .filter(lambda row: row.Medal is not None and row.Medal != "NA") \
    .map(lambda row: (row.NOC, row.Sport)) \
    .sample(False, 0.05, seed=42)

# zeroValue: (count=0, set_of_sports=set())
# seqFunc: на каждой партиции добавляем спорт в множество и +1 к count
# combFunc: объединяем множества и складываем count
result = medal_rdd.aggregateByKey(
    (0, set()),
    lambda acc, sport: (acc[0] + 1, acc[1] | {sport}),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] | acc2[1])
)

for noc, (count, sports) in result.sortBy(lambda x: x[1][0], ascending=False).take(5):
    print(f"  {noc}: {count} медалей, {len(sports)} видов спорта")

  USA: 302 медалей, 40 видов спорта
  URS: 130 медалей, 25 видов спорта
  FRA: 97 медалей, 31 видов спорта
  GBR: 95 медалей, 29 видов спорта
  GER: 94 медалей, 26 видов спорта


### `combineByKey` — самый общий комбинер
Ещё более гибкий, чем `aggregateByKey`. Позволяет по-разному обрабатывать первое значение ключа и последующие.

Параметры:
- `createCombiner` — функция для первого значения каждого ключа
- `mergeValue` — функция для последующих значений
- `mergeCombiners` — функция для объединения комбинеров с разных партиций

Это "строительный блок" — `reduceByKey`, `aggregateByKey` реализованы поверх `combineByKey`.

In [41]:
# combineByKey: средний возраст по виду спорта
age_by_sport = athletes_rdd \
    .filter(lambda row: row.Age is not None and row.Age != "NA") \
    .map(lambda row: (row.Sport, int(row.Age))) \
    .sample(False, 0.1, seed=42)

avg_age = age_by_sport.combineByKey(
    lambda age: (age, 1),
    lambda acc, age: (acc[0] + age, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
)

result = avg_age.mapValues(lambda acc: round(acc[0] / acc[1], 1))
for sport, avg in result.sortBy(lambda x: x[1], ascending=False).take(10):
    print(f"  {sport}: {avg} лет")

  Roque: 64.0 лет
  Art Competitions: 46.4 лет
  Racquets: 45.0 лет
  Alpinism: 42.0 лет
  Equestrianism: 34.6 лет
  Polo: 33.8 лет
  Shooting: 33.1 лет
  Croquet: 33.0 лет
  Curling: 31.9 лет
  Sailing: 30.3 лет


### `countByKey` — подсчёт количества значений по ключу (action)
Возвращает Python dict `{key: count}` на драйвере.
Удобно для быстрого анализа, но результат должен поместиться в память драйвера.

In [42]:
sport_counts = athletes_rdd \
    .map(lambda row: (row.Season, 1)) \
    .countByKey()
print("Атлеты по сезону:", dict(sport_counts))

# countByValue — подсчёт уникальных значений (action)
season_counts = athletes_rdd \
    .map(lambda row: row.Season) \
    .countByValue()
print("Тот же результат через countByValue:", dict(season_counts))

Атлеты по сезону: {'Summer': 222552, 'Winter': 48564}
Тот же результат через countByValue: {'Summer': 222552, 'Winter': 48564}


## Сортировка
### `sortBy` — сортировка по произвольному ключу
### `sortByKey` — сортировка Pair RDD по ключу (если ключ сравним)
### `top(n)` / `takeOrdered(n)` — вернуть n элементов (actions)

In [43]:
# sortBy: отсортировать атлетов по возрасту (убывание)
# Берём подвыборку для скорости
sample_athletes = athletes_rdd.sample(False, 0.01, seed=42)

oldest = sample_athletes \
    .filter(lambda row: row.Age is not None and row.Age != "NA") \
    .sortBy(lambda row: int(row.Age), ascending=False) \
    .map(lambda row: (row.Name, row.Age)) \
    .take(5)
print("Самые возрастные (из выборки):")
for name, age in oldest:
    print(f"  {name}: {age} лет")

# sortByKey: сортировка Pair RDD
pairs = sc.parallelize([(3, "c"), (1, "a"), (2, "b")])
print("sortByKey asc:", pairs.sortByKey().collect())
print("sortByKey desc:", pairs.sortByKey(ascending=False).collect())

# top / takeOrdered — actions, не создают новый RDD
nums = sc.parallelize([3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5])
print("top 3:", nums.top(3))
print("takeOrdered 3:", nums.takeOrdered(3))
print("top 3 по модулю:", nums.top(3, key=abs))

Самые возрастные (из выборки):
  Ernestine Lonie Ernesta Robert-Mrignac: 74 лет
  Sir John Lavery: 72 лет
  Charles William Martin: 71 лет
  Alfred James Munnings: 69 лет
  Alfred James Munnings: 69 лет
sortByKey asc: [(1, 'a'), (2, 'b'), (3, 'c')]
sortByKey desc: [(3, 'c'), (2, 'b'), (1, 'a')]
top 3: [9, 6, 5]
takeOrdered 3: [1, 1, 2]
top 3 по модулю: [9, 6, 5]


## Actions — действия, запускающие вычисления

In [74]:
# collect — собрать все элементы на драйвер (осторожно с большими RDD!)
rdd = sc.parallelize([1, 2, 3])
print("collect:", rdd.collect())

# take(n) — первые n элементов
print("take(2):", rdd.take(2))

# first — первый элемент
print("first:", rdd.first())

# count — общее количество
print("count:", rdd.count())

# countByValue — количество каждого уникального значения
print("countByValue:", sc.parallelize(["a", "b", "a", "c", "a", "b"]).countByValue())

collect: [1, 2, 3]
take(2): [1, 2]
first: 1
count: 3
countByValue: defaultdict(<class 'int'>, {'a': 3, 'b': 2, 'c': 1})


In [45]:
# reduce — свернуть весь RDD в одно значение
rdd = sc.parallelize([1, 2, 3, 4, 5])

print("Сумма (reduce):", rdd.reduce(lambda a, b: a + b))
print("Максимум (reduce):", rdd.reduce(lambda a, b: max(a, b)))
print("Произведение (reduce):", rdd.reduce(lambda a, b: a * b))

# fold — как reduce, но с начальным значением (zero value)
# zero value применяется к каждой партиции отдельно
print("Сумма (fold):", rdd.fold(0, lambda a, b: a + b))
print("Сумма (fold, seed=10):", rdd.fold(10, lambda a, b: a + b))  # 10 * numPartitions + sum

Сумма (reduce): 15
Максимум (reduce): 5
Произведение (reduce): 120
Сумма (fold): 15
Сумма (fold, seed=10): 145


### `aggregate` — самая общная агрегация без ключа
Как `fold`, но позволяет менять тип результата.
`aggregate(zeroValue, seqFunc, combFunc)`

In [46]:
# aggregate: посчитать (sum, count) одновременно
# zeroValue: (sum=0, count=0)
rdd = sc.parallelize([10, 20, 30, 40, 50])

result = rdd.aggregate(
    (0, 0),
    lambda acc, x: (acc[0] + x, acc[1] + 1),       # seqFunc: на каждой партиции
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])  # combFunc: между партициями
)

print(f"Сумма: {result[0]}, Количество: {result[1]}, Среднее: {result[0] / result[1]:.1f}")

# mean, sum, max, min — готовые методы для простых случаев
print("mean:", rdd.mean())
print("sum:", rdd.sum())
print("max:", rdd.max())
print("min:", rdd.min())
print("stdev:", rdd.stdev())
print("variance:", rdd.variance())

# stats — всё сразу (Histogram)
print("stats:", rdd.stats())

Сумма: 150, Количество: 5, Среднее: 30.0
mean: 30.0
sum: 150
max: 50
min: 10
stdev: 14.142135623730951
variance: 200.0
stats: (count: 5, mean: 30.0, stdev: 14.142135623730951, max: 50.0, min: 10.0)


## Set operations — операции над множествами
`union`, `intersection`, `subtract`, `cartesian`.

Внимание: `intersection` и `subtract` требуют shuffle — дорого.

In [47]:
rdd1 = sc.parallelize([1, 2, 3, 4, 5])
rdd2 = sc.parallelize([4, 5, 6, 7, 8])

# union — объединение (с дубликатами!)
print("union:", rdd1.union(rdd2).collect())

# intersection — пересечение (без дубликатов)
print("intersection:", rdd1.intersection(rdd2).collect())

# subtract — разность (элементы rdd1 без элементов rdd2)
print("subtract:", rdd1.subtract(rdd2).collect())

# cartesian — декартово произведение (осторожно: O(n*m)!)
small1 = sc.parallelize([1, 2])
small2 = sc.parallelize(["a", "b", "c"])
print("cartesian:", small1.cartesian(small2).collect())

union: [1, 2, 3, 4, 5, 4, 5, 6, 7, 8]
intersection: [4, 5]
subtract: [1, 2, 3]
cartesian: [(1, 'a'), (1, 'b'), (1, 'c'), (2, 'a'), (2, 'b'), (2, 'c')]


## Join operations на Pair RDD
RDD поддерживают те же типы join, что и SQL: `inner`, `leftOuter`, `rightOuter`, `fullOuter`.
Также есть `cogroup` — группировка значений по ключу из нескольких RDD.

In [48]:
# Создадим два Pair RDD
athletes_small = sc.parallelize([
    (1, "Alice"), (2, "Bob"), (3, "Charlie"), (4, "Diana"), (5, "Eve")
])

medals_small = sc.parallelize([
    (1, "Gold"), (2, "Silver"), (3, "Bronze"), (3, "Gold"), (6, "Silver")
])

# inner join — только ключи, присутствующие в обоих RDD
inner = athletes_small.join(medals_small)
print("inner join:", inner.collect())

# leftOuterJoin — все ключи из левого RDD, None если нет соответствия
left = athletes_small.leftOuterJoin(medals_small)
print("leftOuterJoin:", left.collect())

# rightOuterJoin — все ключи из правого RDD
right = athletes_small.rightOuterJoin(medals_small)
print("rightOuterJoin:", right.collect())

# fullOuterJoin — все ключи из обоих RDD
full = athletes_small.fullOuterJoin(medals_small)
print("fullOuterJoin:", full.collect())

inner join: [(1, ('Alice', 'Gold')), (2, ('Bob', 'Silver')), (3, ('Charlie', 'Bronze')), (3, ('Charlie', 'Gold'))]
leftOuterJoin: [(1, ('Alice', 'Gold')), (2, ('Bob', 'Silver')), (3, ('Charlie', 'Bronze')), (3, ('Charlie', 'Gold')), (4, ('Diana', None)), (5, ('Eve', None))]
rightOuterJoin: [(1, ('Alice', 'Gold')), (2, ('Bob', 'Silver')), (3, ('Charlie', 'Bronze')), (3, ('Charlie', 'Gold')), (6, (None, 'Silver'))]
fullOuterJoin: [(1, ('Alice', 'Gold')), (2, ('Bob', 'Silver')), (3, ('Charlie', 'Bronze')), (3, ('Charlie', 'Gold')), (4, ('Diana', None)), (5, ('Eve', None)), (6, (None, 'Silver'))]


In [49]:
# cogroup — группировка значений из нескольких RDD по ключу
# Возвращает (key, (iterable_values_rdd1, iterable_values_rdd2))
grouped = athletes_small.cogroup(medals_small)
result = grouped.mapValues(lambda vs: (list(vs[0]), list(vs[1]))).collect()
for key, (names, medals) in sorted(result):
    print(f"  key={key}: names={names}, medals={medals}")

  key=1: names=['Alice'], medals=['Gold']
  key=2: names=['Bob'], medals=['Silver']
  key=3: names=['Charlie'], medals=['Bronze', 'Gold']
  key=4: names=['Diana'], medals=[]
  key=5: names=['Eve'], medals=[]
  key=6: names=[], medals=['Silver']


## Partitioning — управление партициями

Количество партиций определяет параллелизм. Правило большого пальца: `2-4 партиции на каждое ядро CPU`.

- `getNumPartitions()` — текущее количество
- `repartition(n)` — перераспределить на n партиций (**всегда shuffle**)
- `coalesce(n)` — уменьшить количество партиций **без shuffle** (только если n < current)
- `glom()` — собрать каждую партицию в список (для отладки)
- `partitionBy(n)` — для Pair RDD, использует HashPartitioner

**`repartition` vs `coalesce`:**
- `repartition` всегда shuffle — дорого, но можно и увеличить, и уменьшить
- `coalesce` без shuffle — дёшево, но только для уменьшения; для увеличения использует shuffle

In [50]:
# Создаём RDD с 5 партициями
rdd = sc.parallelize(range(100), numSlices=5)
print("Исходные партиции:", rdd.getNumPartitions())

# Смотрим размер каждой партиции через glom
partition_sizes = rdd.glom().map(len).collect()
print("Размеры партиций:", partition_sizes)

# repartition: увеличиваем до 10 (shuffle!)
rdd_more = rdd.repartition(10)
print("После repartition(10):", rdd_more.getNumPartitions())
print("Размеры:", rdd_more.glom().map(len).collect())

# coalesce: уменьшаем до 2 (без shuffle)
rdd_fewer = rdd.coalesce(2)
print("После coalesce(2):", rdd_fewer.getNumPartitions())
print("Размеры:", rdd_fewer.glom().map(len).collect())

# coalesce с shuffle=True — ведёт себя как repartition
rdd_force = rdd_fewer.coalesce(5, shuffle=True)
print("После coalesce(5, shuffle=True):", rdd_force.getNumPartitions())
print("Размеры:", rdd_force.glom().map(len).collect())

Исходные партиции: 5
Размеры партиций: [20, 20, 20, 20, 20]
После repartition(10): 10
Размеры: [20, 0, 0, 0, 20, 20, 0, 0, 10, 30]
После coalesce(2): 2
Размеры: [40, 60]
После coalesce(5, shuffle=True): 5
Размеры: [20, 20, 20, 10, 30]


In [ ]:
# partitionBy — для Pair RDD, использует HashPartitioner
# Полезно перед многократными join/reduceByKey по одному ключу
pairs = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("c", 4), ("b", 5)])

# Без partitionBy: партиции распределяются случайно
print("Без partitionBy:", pairs.getNumPartitions())

# partitionBy: все элементы с одним ключом — в одной партиции
partitioned = pairs.partitionBy(3)
print("После partitionBy(3):", partitioned.getNumPartitions())

# Проверим: какие ключи в какой партиции
for i, part in enumerate(partitioned.glom().collect()):
    print(f"  Партиция {i}: {part}")

## `mapPartitions` / `mapPartitionsWithIndex` — обработка по партициям

`map` работает с одним элементом за раз.
`mapPartitions` работает с **целой партицией** (итератором) за раз.

**Когда использовать `mapPartitions`:**
- Создание дорогих объектов (DB-соединения, ML-модели) — создаёте один раз на партицию, а не на каждый элемент
- Агрегации внутри партиции
- Пакетная обработка

In [ ]:
# mapPartitions: обработка целой партиции за раз
rdd = sc.parallelize(range(1, 21), numSlices=4)

# Каждый элемент → x²
print("map:", rdd.map(lambda x: x ** 2).collect())

# mapPartitions: итератор партиции → итератор результатов
# Можно вернуть ДРУГОЕ количество элементов!
def process_partition(iterator):
    """Посчитать сумму элементов партиции и вернуть одно значение"""
    yield sum(iterator)

print("mapPartitions (сумма каждой партиции):",
      rdd.mapPartitions(process_partition).collect())

# mapPartitionsWithIndex — то же, но с индексом партиции
def process_with_index(idx, iterator):
    yield f"Партиция {idx}: {list(iterator)}"

print("mapPartitionsWithIndex:")
for item in rdd.mapPartitionsWithIndex(process_with_index).collect():
    print(" ", item)

## Persistence: `cache` / `persist` / `unpersist`

По умолчанию RDD пересчитывается при каждом action. Если RDD используется несколько раз — закешируйте его.

Уровни хранения (`pyspark.StorageLevel`):
- `MEMORY_ONLY` — только в RAM (default для cache)
- `MEMORY_AND_DISK` — в RAM, излишек на диск
- `DISK_ONLY` — только на диск
- `MEMORY_ONLY_SER` — сериализованные объекты в RAM (меньше памяти)
- `NONE` — не хранить

In [ ]:
from pyspark import StorageLevel

rdd = sc.parallelize(range(1000))

# cache = persist(MEMORY_AND_DISK)
rdd_cached = rdd.cache()

# persist с конкретным уровнем
rdd.persist(StorageLevel.MEMORY_ONLY)

# Теперь повторные действия используют кеш, а не пересчитывают
print("count:", rdd.count())         # вычисляет и кеширует
print("sum:", rdd.sum())             # берёт из кеша
print("max:", rdd.max())             # берёт из кеша

# getStorageLevel — текущий уровень
print("Storage level:", rdd.getStorageLevel())

# unpersist — освободить память
rdd.unpersist()
print("После unpersist:", rdd.getStorageLevel())

## Broadcast Variables — широковещательные переменные

Позволяют эффективно отправить **read-only** значение всем узлам (один раз), а не с каждой задачей.

**Когда использовать:**
- Справочник для join с маленькой таблицей
- Параметры конфигурации
- ML-модель для inference

Без broadcast: Spark отправляет копию переменной с каждой задачей.
С broadcast: переменная отправляется на каждый узел один раз и кешируется.

In [ ]:
# Broadcast: справочник для обогащения данных

# Справочник: NOC → полное название страны
noc_to_country = {
    "USA": "United States",
    "RUS": "Russia",
    "CHN": "China",
    "GBR": "Great Britain",
    "GER": "Germany",
    "FRA": "France",
    "JPN": "Japan",
}

# Отправляем справочник всем узлам через broadcast
broadcast_noc = sc.broadcast(noc_to_country)

# Используем в map: value доступна на каждом узле без пересылки
enriched = athletes_rdd \
    .filter(lambda row: row.NOC in broadcast_noc.value) \
    .map(lambda row: (row.Name, row.NOC, broadcast_noc.value[row.NOC])) \
    .distinct() \
    .take(10)

for name, noc, country in enriched:
    print(f"  {name} ({noc}) → {country}")

# Освобождаем broadcast
broadcast_noc.unpersist()

## Accumulators — аккумуляторы

Переменные, которые можно **только увеличивать** (add) на узлах и читать на драйвере.
Используются для счётчиков событий (количество ошибок, обработанных записей и т.д.).

Важно: аккумуляторы **надёжны только в actions** (не transformations). В transformations Spark может пересчитать задачу, и аккумулятор увеличится повторно.

In [ ]:
# Accumulator: подсчёт строк с пустыми значениями
empty_medal_counter = sc.accumulator(0)

def count_empty_medals(row):
    if row.Medal is None or row.Medal == "NA":
        empty_medal_counter.add(1)
    return row

# Применяем в action (foreach) — надёжно
athletes_rdd.sample(False, 0.1, seed=42).foreach(count_empty_medals)

print(f"Строк без медали (в 10% выборки): {empty_medal_counter.value}")

# float accumulator
total_age = sc.accumulator(0.0)
athletes_rdd \
    .sample(False, 0.05, seed=42) \
    .filter(lambda row: row.Age is not None and row.Age != "NA") \
    .foreach(lambda row: total_age.add(float(row.Age)))

count = athletes_rdd.sample(False, 0.05, seed=42) \
    .filter(lambda row: row.Age is not None and row.Age != "NA").count()
print(f"Сумма возрастов (5% выборки): {total_age.value}, count: {count}, avg: {total_age.value / count:.1f}")

## `foreach` — действие для side-effects

`foreach(fn)` применяет функцию к каждому элементу, но ничего не возвращает. Используется для side-effects: запись в БД, отправка сообщений, логирование.

В отличие от `map`, результат не возвращается на драйвер — функция выполняется на executor-ах.

In [ ]:
# foreach: пример с "записью в лог"
# (здесь просто печатаем, но в реальности — запись в БД/Kafka)

log_lines = []  # ВНИМАНИЕ: это только для демонстрации!
# В реальном кластере этот список будет пуст — foreach выполняется на executor-ах

def log_athlete(row):
    # Здесь могла бы быть запись в БД
    pass

# В локальном режиме работает (shared memory), в кластере — нет
athletes_rdd.sample(False, 0.001, seed=42).foreach(log_athlete)
print("foreach выполнен (в локальном режиме side-effects видны на драйвере)")

# Для сбора данных используйте collect/take, а не глобальные переменные
# foreach — только для записи наружу (БД, Kafka, файлы)

## `pipe` — вызов внешних команд

`pipe(command)` — пропускает каждый элемент RDD через внешнюю команду (shell-скрипт, Python-скрипт и т.д.).
Полезно для интеграции с legacy-инструментами.

In [ ]:
# pipe: пропустить элементы через shell-команду
rdd = sc.parallelize(["hello", "world", "apache", "spark"])

# Каждый элемент передаётся на stdin команды
result = rdd.pipe("tr a-z A-Z").collect()
print("После pipe (uppercase):", result)

# pipe с wc -l — посчитать строки на каждой партиции
lengths = rdd.pipe("wc -l").collect()
print("wc -l:", lengths)

## Сохранение RDD

`saveAsTextFile(path)` — сохранить RDD как текстовые файлы (один файл на партицию).
Путь может быть локальным или HDFS.

Внимание: создаётся **директория** с файлами part-00000, part-00001, ...

In [ ]:
# saveAsTextFile — сохранение в HDFS
result_rdd = athletes_rdd \
    .filter(lambda row: row.Medal is not None and row.Medal != "NA") \
    .map(lambda row: f"{row.Name},{row.Team},{row.Medal}") \
    .sample(False, 0.01, seed=42)

# Сохраняем в HDFS
! hdfs dfs -rm -r -f /user/spark_advanced/medalists_output
result_rdd.saveAsTextFile("/user/spark_advanced/medalists_output")

# Проверяем результат
! hdfs dfs -ls /user/spark_advanced/medalists_output/
print("---")
! hdfs dfs -head /user/spark_advanced/medalists_output/part-00000

## Lineage и DAG

RDD хранит **lineage** — граф всех преобразований, из которых он получен. При сбое узла Spark может пересчитать потерянные партиции из исходных данных.

`toDebugString()` — посмотреть lineage (DAG) RDD.

In [ ]:
# toDebugString — посмотреть lineage RDD
rdd = athletes_rdd \
    .filter(lambda row: row.Medal is not None and row.Medal != "NA") \
    .map(lambda row: (row.NOC, 1)) \
    .reduceByKey(lambda a, b: a + b)

print(rdd.toDebugString().decode("utf-8"))

# Видим цепочку: ParallelCollectionRDD → MapPartitionsRDD → MapPartitionsRDD → ShuffledRDD → MapPartitionsRDD
# ShuffledRDD — точка shuffle (при reduceByKey)

## Практика на athletes: комплексные примеры
Соберём вместе всё изученное.

In [ ]:
# Пример 1: Топ-3 самых возрастных медалиста по каждому полу
top_oldest = athletes_rdd \
    .filter(lambda row: row.Medal is not None and row.Medal != "NA" and row.Age is not None and row.Age != "NA") \
    .map(lambda row: (row.Gender, (row.Name, int(row.Age)))) \
    .groupByKey() \
    .flatMapValues(lambda vals: sorted(vals, key=lambda x: -x[1])[:3])

for gender, (name, age) in top_oldest.collect():
    print(f"  [{gender}] {name}: {age} лет")

In [ ]:
# Пример 2: Средний возраст медалистов по виду медали (через aggregateByKey)
avg_by_medal = athletes_rdd \
    .filter(lambda row: row.Medal is not None and row.Medal != "NA" and row.Age is not None and row.Age != "NA") \
    .map(lambda row: (row.Medal, int(row.Age))) \
    .aggregateByKey(
    (0, 0),
    lambda acc, age: (acc[0] + age, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
) \
    .mapValues(lambda acc: round(acc[0] / acc[1], 1))

for medal, avg in avg_by_medal.collect():
    print(f"  {medal}: средний возраст {avg} лет")

In [ ]:
# Пример 3: Найти пары спортсменов из одной команды с одинаковым возрастом
# (на маленькой выборке)
sample = athletes_rdd \
    .filter(lambda row: row.Age is not None and row.Age != "NA") \
    .map(lambda row: (row.Team, row.Age, row.Name)) \
    .sample(False, 0.005, seed=42) \
    .collect()

# Группируем по (Team, Age) и ищем группы с >1 атлетом
from collections import defaultdict
groups = defaultdict(list)
for team, age, name in sample:
    groups[(team, age)].append(name)

count = 0
for (team, age), names in groups.items():
    if len(names) > 1:
        print(f"  {team}, возраст {age}: {names[:5]}")
        count += 1
        if count >= 10:
            break

print(f"\nВсего найдено пар (в 0.5% выборки): {sum(1 for v in groups.values() if len(v) > 1)}")

In [ ]:
# Пример 4: Word Count на столбце "Sport" через RDD
# Классический pipeline: map → reduceByKey → sortBy
sport_counts = athletes_rdd \
    .map(lambda row: (row.Sport, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: -x[1])

print("Топ-10 видов спорта по количеству атлетов:")
for sport, count in sport_counts.take(10):
    print(f"  {sport}: {count}")

print(f"\nВсего уникальных видов спорта: {sport_counts.count()}")

In [ ]:
# Пример 5: Self-join на RDD
# Найти атлетов из одной команды, участвовавших в один год
pairs_rdd = athletes_rdd \
    .filter(lambda row: row.Team is not None and row.Year is not None) \
    .map(lambda row: ((row.Team, int(row.Year)), row.Name)) \
    .sample(False, 0.01, seed=42)

# Группируем по (Team, Year) — получаем список имён
grouped = pairs_rdd.groupByKey().mapValues(lambda names: list(names))

# Берём только группы с >1 атлетом
teammates = grouped.filter(lambda x: len(x[1]) > 1)

count = 0
for (team, year), names in teammates.take(10):
    print(f"  {team} ({year}): {names[:5]}")
    count += 1

print(f"\nВсего групп однокомандников (в 1% выборки): {teammates.count()}")

## Конвертация RDD ↔ DataFrame

RDD → DataFrame: `rdd.toDF([schema])` или `spark.createDataFrame(rdd, schema)`.
DataFrame → RDD: `df.rdd`.

Используйте DataFrame, когда важна производительность и оптимизация.
Используйте RDD, когда нужен тонкий контроль или данные неструктурированные.

In [ ]:
# RDD → DataFrame
pairs_rdd = athletes_rdd \
    .filter(lambda row: row.Medal is not None and row.Medal != "NA") \
    .map(lambda row: (row.NOC, row.Sport, row.Medal))

# Способ 1: toDF с именами колонок
df_from_rdd = pairs_rdd.toDF(["NOC", "Sport", "Medal"])
df_from_rdd.show(5)

# Способ 2: createDataFrame с schema
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([
    StructField("noc", StringType(), True),
    StructField("sport", StringType(), True),
    StructField("medal", StringType(), True),
])
df_from_rdd2 = spark.createDataFrame(pairs_rdd, schema)
df_from_rdd2.printSchema()
df_from_rdd2.show(5)

# DataFrame → RDD (обратно)
rdd_from_df = df_from_rdd.rdd
print("Тип:", type(rdd_from_df))
print("Первые 3:", rdd_from_df.take(3))

## Итоги

**RDD** — низкоуровневый API Spark. Ключевые моменты:

1. **Transformations** (`map`, `filter`, `flatMap`, `reduceByKey`, ...) — ленивые, строят lineage
2. **Actions** (`collect`, `count`, `take`, `reduce`, ...) — запускают вычисления
3. **Pair RDD** — `(key, value)`, поддерживают `reduceByKey`, `join`, `groupByKey`, `aggregateByKey`
4. **`reduceByKey` > `groupByKey`** — всегда, когда возможно
5. **Partitioning** — `repartition` (shuffle) vs `coalesce` (без shuffle)
6. **Persistence** — `cache`/`persist` для повторно используемых RDD
7. **Broadcast** — для read-only справочников
8. **Accumulators** — для счётчиков (только в actions!)
9. **Lineage** — восстановление при сбое через `toDebugString()`

**В современной разработке предпочитайте DataFrame** — Catalyst optimizer + Tungsten дают значительный прирост производительности. RDD — когда нужен полный контроль.

In [ ]:
spark.stop()